# Thu thập dữ liệu ô tô từ Chợ Tốt

**Nguồn:** danh sách tin rao bán ô tô tại `https://xe.chotot.com/mua-ban-oto`, mỗi tin lấy 12 thuộc tính (số km, xuất xứ, tình trạng, bảo hành, hãng, dòng xe, năm sản xuất, hộp số, nhiên liệu, kiểu dáng, số chỗ, giá) cùng URL. Kết quả: `raw_data_oto.csv` (14.835 tin).

**Thời điểm thu thập:** file `raw_data_oto.csv` được ghi lần cuối ngày 25/10/2026. Giá xe thay đổi theo thời gian nên báo cáo cần ghi rõ ngày crawl.

**Thu thập có trách nhiệm:**
- Chỉ lấy thông tin công khai trên trang tin rao, không đăng nhập, không lấy thông tin liên hệ của người bán; dữ liệu chỉ dùng cho mục đích học tập.
- Nghỉ giữa các yêu cầu (`time.sleep` 0,5 giây giữa các tin, 1 giây giữa các trang) và tự lùi thời gian khi bị trả về mã 403/429, để không gây tải cho máy chủ.
- Trước khi chạy lại cần xem `https://xe.chotot.com/robots.txt` và điều khoản sử dụng của Chợ Tốt.

**Hạn chế:** danh sách tin được tìm bằng class CSS `crd7gu7`. Đây là tên class sinh tự động, nên khi trang đổi giao diện thì code có thể không còn tìm thấy tin nào và cần sửa lại bộ chọn.

## 0. Thử nghiệm (nháp)
Cell dưới đây chỉ **thử** lấy URL tin đăng từ 5 trang đầu để kiểm tra bộ chọn CSS. Cell này không ghi file và không cần chạy lại.

In [2]:
import time
from urllib.parse import urljoin
import requests
from bs4 import BeautifulSoup

base_domain = "https://xe.chotot.com"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

start_page = 1
end_page = 5  # Thay đổi số trang bạn muốn cào
total_count = 0

for page in range(start_page, end_page + 1):
    url = f"https://xe.chotot.com/mua-ban-oto?page={page}"
    print(f"\n--- TRANG {page} ({url}) ---")

    try:
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            print(f"Lỗi truy cập trang {page}: Mã trạng thái {response.status_code}")
            continue

        soup = BeautifulSoup(response.text, "html.parser")

        # Tìm các thẻ div chứa thẻ <a> có thuộc tính href[cite: 1]
        div_items = soup.find_all("div", class_="crd7gu7")

        page_count = 0
        for div in div_items:
            a_tag = div.find("a", href=True)
            if a_tag:
                total_count += 1
                page_count += 1
                # Nối link tương đối với domain chính
                full_url = urljoin(base_domain, a_tag["href"])
                print(f"[{total_count}] {full_url}")

        if page_count == 0:
            print("Không tìm thấy URL nào trên trang này.")

        # Nghỉ 1.5 giây giữa các trang để tránh bị chặn IP
        time.sleep(1.5)

    except Exception as e:
        print(f"Lỗi khi xử lý trang {page}: {e}")

print(f"\n=> Tổng cộng đã tìm thấy {total_count} URL xe.")


--- TRANG 1 (https://xe.chotot.com/mua-ban-oto?page=1) ---
[1] https://xe.chotot.com/mua-ban-oto-quan-1-tp-ho-chi-minh/135108889.htm#px=SR-stickyad-[PO-1][PL-top]
[2] https://xe.chotot.com/mua-ban-oto-quan-thanh-khe-da-nang/135321478.htm#px=SR-stickyad-[PO-2][PL-top]
[3] https://xe.chotot.com/mua-ban-oto-thanh-pho-thu-duc-tp-ho-chi-minh/131953096.htm#px=SR-stickyad-[PO-3][PL-top]
[4] https://xe.chotot.com/mua-ban-oto-quan-7-tp-ho-chi-minh/135085489.htm#px=SR-stickyad-[PO-4][PL-top]
[5] https://xe.chotot.com/mua-ban-oto-quan-binh-thuy-can-tho/135331414.htm#px=SR-stickyad-[PO-5][PL-top]
[6] https://xe.chotot.com/mua-ban-oto-huyen-hoc-mon-tp-ho-chi-minh/135455024.htm
[7] https://xe.chotot.com/mua-ban-oto-quan-7-tp-ho-chi-minh/135455295.htm
[8] https://xe.chotot.com/mua-ban-oto-quan-8-tp-ho-chi-minh/135455294.htm
[9] https://xe.chotot.com/mua-ban-oto-huyen-binh-chanh-tp-ho-chi-minh/135263817.htm
[10] https://xe.chotot.com/mua-ban-oto-huyen-nha-be-tp-ho-chi-minh/135455253.htm
[11] https://

## 1. Thu thập chính
Duyệt 750 trang danh sách, vào từng tin để lấy các thuộc tính, ghi nối tiếp vào `raw_data_oto.csv`. Có cơ chế **chạy tiếp**: các URL đã có trong file sẽ được bỏ qua, nên nếu bị ngắt giữa chừng thì chỉ cần chạy lại cell.

In [4]:
import csv
import os
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urldefrag

base_domain = "https://xe.chotot.com"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

# 12 đặc trưng, đúng thứ tự header; 'price' lấy từ thẻ name="price" -> value
ids = [
    'mileage_v2', 'carorigin', 'condition_ad', 'veh_warranty_policy',
    'carbrand', 'carmodel', 'mfdate', 'gearbox', 'fuel', 'cartype', 'carseats',
    'price'
]

start_page = 1
end_page = 750
csv_filename = "raw_data_oto.csv"

session = requests.Session()
session.headers.update(headers)


def get_with_retry(url, retries=3, timeout=15):
    for attempt in range(retries):
        try:
            res = session.get(url, timeout=timeout)
            if res.status_code == 200:
                return res
            if res.status_code in (403, 429):
                time.sleep(5 * (attempt + 1))
        except Exception:
            time.sleep(2 * (attempt + 1))
    return None


def extract_value(soup, item_id):
    if item_id == "price":
        elem = soup.find(attrs={"name": "price"})
        if elem and elem.get("value"):
            return elem["value"]
        elem = soup.find(attrs={"itemprop": "price"})  # dự phòng
    else:
        elem = soup.find(attrs={"itemprop": item_id})
    if elem:
        return elem.get("content") or elem.get("value") or elem.get_text(strip=True) or "N/A"
    return "N/A"


# Resume: đọc các URL đã cào từ lần chạy trước
done_urls = set()
file_exists = os.path.exists(csv_filename)
if file_exists:
    with open(csv_filename, encoding="utf-8-sig") as f:
        done_urls = {r["url"] for r in csv.DictReader(f)}
    print(f"Tiếp tục: đã có {len(done_urls)} xe trong file.")

seen = set(done_urls)
no = len(done_urls)  # số thứ tự tiếp tục từ lần chạy trước
saved = 0

with open(csv_filename, mode="a", newline="", encoding="utf-8-sig") as file:
    writer = csv.writer(file)
    if not file_exists:
        writer.writerow(["no", "url"] + ids)

    for page in range(start_page, end_page + 1):
        res = get_with_retry(f"{base_domain}/mua-ban-oto?page={page}")
        if res is None:
            print(f"Trang {page}: lỗi, bỏ qua.")
            continue

        soup = BeautifulSoup(res.text, "html.parser")
        page_urls = []
        for div in soup.find_all("div", class_="crd7gu7"):
            a_tag = div.find("a", href=True)
            if a_tag:
                u = urldefrag(urljoin(base_domain, a_tag["href"]))[0]  # bỏ #px=...
                if u not in seen:
                    seen.add(u)
                    page_urls.append(u)

        for car_url in page_urls:
            r = get_with_retry(car_url)
            if r is None:
                print(f"  Lỗi: {car_url}")
                continue
            s = BeautifulSoup(r.text, "html.parser")
            no += 1
            writer.writerow([no, car_url] + [extract_value(s, i) for i in ids])
            file.flush()
            saved += 1
            time.sleep(0.5)

        print(f"Trang {page}/{end_page}: +{len(page_urls)} xe (tổng mới: {saved})")
        time.sleep(1)

print(f"\nHoàn tất! Đã lưu {saved} xe mới vào '{csv_filename}'.")


Trang 1/750: +25 xe (tổng mới: 25)
Trang 2/750: +25 xe (tổng mới: 50)
Trang 3/750: +19 xe (tổng mới: 69)
Trang 4/750: +20 xe (tổng mới: 89)
Trang 5/750: +20 xe (tổng mới: 109)
Trang 6/750: +19 xe (tổng mới: 128)
Trang 7/750: +17 xe (tổng mới: 145)
Trang 8/750: +19 xe (tổng mới: 164)
Trang 9/750: +20 xe (tổng mới: 184)
Trang 10/750: +20 xe (tổng mới: 204)
Trang 11/750: +19 xe (tổng mới: 223)
Trang 12/750: +20 xe (tổng mới: 243)
Trang 13/750: +18 xe (tổng mới: 261)
Trang 14/750: +20 xe (tổng mới: 281)
Trang 15/750: +19 xe (tổng mới: 300)
Trang 16/750: +19 xe (tổng mới: 319)
Trang 17/750: +20 xe (tổng mới: 339)
Trang 18/750: +20 xe (tổng mới: 359)
Trang 19/750: +19 xe (tổng mới: 378)
Trang 20/750: +20 xe (tổng mới: 398)
Trang 21/750: +19 xe (tổng mới: 417)
Trang 22/750: +18 xe (tổng mới: 435)
Trang 23/750: +19 xe (tổng mới: 454)
Trang 24/750: +18 xe (tổng mới: 472)
Trang 25/750: +20 xe (tổng mới: 492)
Trang 26/750: +19 xe (tổng mới: 511)
Trang 27/750: +20 xe (tổng mới: 531)
Trang 28/750: 